# Step 4 — Random Forest Model Experimentation & Optimization

**Project:** AI Operations Assistant for Smart Inventory & Billing  
**Target:** Future 7-Day Product Demand  
**Previous best reference:** Random Forest MAE = **26.4215** (from the previous V3 notebook; this notebook will recalculate results).

## Objectives
1. Rebuild the same leakage-aware dataset and features used in Notebook 03.
2. Create a chronological training/validation split with a 7-day purge.
3. Compare controlled Random Forest configurations using validation data only.
4. Select the best configuration using validation MAE.
5. Refit the selected model on all pre-test training data.
6. Evaluate the selected model once on the untouched final test period.
7. Save experiment results, final metrics, predictions and the selected model.

> **Important:** Do not use the final test set to choose hyperparameters. The validation period is used for experimentation; the final test period is reserved for the final evaluation.

## Before running

- Select your normal **Anaconda Python kernel** (the same one used for Notebook 03).
- Keep `processed_daily_sales.csv` and `Online Retail.xlsx` in the project's `data` folder.
- Run the notebook from the first cell, using **Restart Kernel → Run All**.
- The Random Forest experiments can take time because the dataset is large. Start with the compact experiment list in Section 10.

In [ ]:
# 1. IMPORT LIBRARIES

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Libraries imported successfully.")

## 2. Configure project paths

The default paths match the project location used in Notebook 03. If your project folder is somewhere else, edit `PROJECT_DIR` only.

Expected files:
- `data/processed_daily_sales.csv`
- `data/Online Retail.xlsx`

In [ ]:
# Portable path resolution (works from notebooks/ or workspace root)
PROJECT_DIR = Path("..") if Path.cwd().name == "notebooks" else Path("smart-inventory-ai") if Path("smart-inventory-ai").exists() else Path(".")

DATA_PATH = PROJECT_DIR / "data" / "processed_daily_sales.csv"
RAW_DATA_PATH = PROJECT_DIR / "data" / "Online Retail.xlsx"

RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Processed data exists:", DATA_PATH.exists(), DATA_PATH.resolve())
print("Raw transaction file exists:", RAW_DATA_PATH.exists(), RAW_DATA_PATH.resolve())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed file not found: {DATA_PATH.resolve()}\n"
        "Ensure data/processed_daily_sales.csv exists."
    )

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw Excel file not found: {RAW_DATA_PATH.resolve()}\n"
        "Ensure data/Online Retail.xlsx exists."
    )

## 3. Load processed daily sales and recreate the target

For each product and date **T**:

`Future7DayDemand = UnitsSold(T+1) + ... + UnitsSold(T+7)`

The target is recreated to avoid using an old or stale target column.

In [ ]:
daily_sales = pd.read_csv(
    DATA_PATH,
    dtype={"StockCode": str},
    low_memory=False
)

daily_sales["Date"] = pd.to_datetime(daily_sales["Date"], errors="coerce")
daily_sales["StockCode"] = daily_sales["StockCode"].astype(str).str.strip()
daily_sales["UnitsSold"] = pd.to_numeric(daily_sales["UnitsSold"], errors="coerce")

daily_sales = daily_sales.dropna(
    subset=["Date", "StockCode", "UnitsSold"]
).copy()

daily_sales = daily_sales.sort_values(
    ["StockCode", "Date"]
).reset_index(drop=True)

target_column = "Future7DayDemand"

daily_sales[target_column] = (
    daily_sales.groupby("StockCode")["UnitsSold"]
    .transform(
        lambda x: x.shift(-1)
                   .rolling(window=7, min_periods=7)
                   .sum()
                   .shift(-6)
    )
)

before = len(daily_sales)
daily_sales = daily_sales.dropna(subset=[target_column]).copy()
print(f"Loaded rows: {before:,}")
print(f"Rows removed for incomplete 7-day target: {before - len(daily_sales):,}")
print(f"Valid target rows: {len(daily_sales):,}")
print("Remaining target NaNs:", daily_sales[target_column].isna().sum())

assert daily_sales[target_column].isna().sum() == 0

## 4. Recreate historical and calendar features

All lag, rolling and historical-mean features use earlier product sales only. The current day's sales are excluded from rolling/history features.

In [ ]:
# Lag features
for lag in [1, 7, 30]:
    daily_sales[f"Lag_{lag}"] = (
        daily_sales.groupby("StockCode")["UnitsSold"].shift(lag)
    )

# Rolling means: shift by one day before calculating
daily_sales["RollingMean_7"] = (
    daily_sales.groupby("StockCode")["UnitsSold"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)

daily_sales["RollingMean_30"] = (
    daily_sales.groupby("StockCode")["UnitsSold"]
    .transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())
)

# Historical mean: past observations only
daily_sales["HistoricalMean"] = (
    daily_sales.groupby("StockCode")["UnitsSold"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Calendar features
daily_sales["DayOfWeek"] = daily_sales["Date"].dt.dayofweek
daily_sales["Month"] = daily_sales["Date"].dt.month
daily_sales["Year"] = daily_sales["Date"].dt.year

print("Historical and calendar features created.")

## 5. Recreate average unit price feature

The price is calculated from cleaned UK transactions. Missing prices are forward-filled within each product only; we do not back-fill from future prices.

In [ ]:
raw_df = pd.read_excel(RAW_DATA_PATH)

raw_df["InvoiceDate"] = pd.to_datetime(raw_df["InvoiceDate"], errors="coerce")
raw_df["Country"] = raw_df["Country"].astype(str).str.strip()
raw_df["StockCode"] = raw_df["StockCode"].astype(str).str.strip()

raw_df = raw_df[raw_df["Country"] == "United Kingdom"].copy()
raw_df = raw_df[~raw_df["InvoiceNo"].astype(str).str.startswith("C")].copy()
raw_df = raw_df[
    (raw_df["Quantity"] > 0) &
    (raw_df["UnitPrice"] > 0)
].copy()

raw_df = raw_df.dropna(subset=["InvoiceDate", "StockCode"])
raw_df = raw_df.drop_duplicates()
raw_df["Date"] = raw_df["InvoiceDate"].dt.normalize()

daily_price = (
    raw_df.groupby(["StockCode", "Date"])["UnitPrice"]
    .mean()
    .reset_index()
    .rename(columns={"UnitPrice": "AvgUnitPrice"})
)

daily_sales = daily_sales.drop(
    columns=[
        c for c in ["AvgUnitPrice", "AvgUnitPrice_x", "AvgUnitPrice_y"]
        if c in daily_sales.columns
    ]
)

daily_sales = daily_sales.merge(
    daily_price,
    on=["StockCode", "Date"],
    how="left",
    validate="many_to_one"
)

daily_sales["AvgUnitPrice"] = (
    daily_sales.groupby("StockCode")["AvgUnitPrice"]
    .transform(lambda x: x.ffill())
)

print("Price feature merged.")
print("Missing price values before split:", daily_sales["AvgUnitPrice"].isna().sum())

## 6. Define features and chronological split

We preserve the V3 final holdout period:
- Model-development data ends: **2011-09-12**
- Purge gap: **2011-09-13 to 2011-09-19**
- Final test starts: **2011-09-20**

Inside the development period, we create a second chronological split:
- Inner training ends: **2011-08-15**
- Inner purge gap: **2011-08-16 to 2011-08-22**
- Validation starts: **2011-08-23**

The 7-day purge prevents training target windows from extending into the next evaluation period.

In [ ]:
feature_columns = [
    "Lag_1",
    "Lag_7",
    "Lag_30",
    "RollingMean_7",
    "RollingMean_30",
    "HistoricalMean",
    "DayOfWeek",
    "Month",
    "Year",
    "AvgUnitPrice"
]

# Final untouched test period (same date boundary as Notebook 03 V3)
DEV_END = pd.Timestamp("2011-09-12")
TEST_START = pd.Timestamp("2011-09-20")

# Inner chronological validation split
INNER_TRAIN_END = pd.Timestamp("2011-08-15")
VALIDATION_START = pd.Timestamp("2011-08-23")

development = daily_sales[daily_sales["Date"] <= DEV_END].copy()
final_test = daily_sales[daily_sales["Date"] >= TEST_START].copy()

inner_train = development[
    development["Date"] <= INNER_TRAIN_END
].copy()

validation = development[
    development["Date"] >= VALIDATION_START
].copy()

print("Inner training:", len(inner_train), inner_train["Date"].min(), "to", inner_train["Date"].max())
print("Validation    :", len(validation), validation["Date"].min(), "to", validation["Date"].max())
print("Final test    :", len(final_test), final_test["Date"].min(), "to", final_test["Date"].max())

assert inner_train["Date"].max() < validation["Date"].min()
assert (validation["Date"].min() - inner_train["Date"].max()).days == 8
assert development["Date"].max() == DEV_END
assert final_test["Date"].min() == TEST_START

print("\nChronological split and 7-day purge boundaries verified.")

## 7. Prepare features using training-only fallback values

Feature fallback statistics are calculated from the relevant training partition only. Targets are never imputed.

The same preparation function is used for validation and final test data.

In [ ]:
LAG_COLUMNS = ["Lag_1", "Lag_7", "Lag_30"]
ROLLING_COLUMNS = ["RollingMean_7", "RollingMean_30"]

def prepare_feature_partitions(train_df, other_dfs, feature_cols):
    """Fill feature NaNs using statistics calculated from train_df only."""
    train_df = train_df.copy()
    other_dfs = [df.copy() for df in other_dfs]

    sales_fallback = train_df["UnitsSold"].mean()
    price_values = train_df["AvgUnitPrice"].dropna()
    price_fallback = price_values.mean() if len(price_values) else 0.0

    for col in LAG_COLUMNS:
        train_df[col] = train_df[col].fillna(0)
        for df in other_dfs:
            df[col] = df[col].fillna(0)

    for col in ROLLING_COLUMNS + ["HistoricalMean"]:
        train_df[col] = train_df[col].fillna(sales_fallback)
        for df in other_dfs:
            df[col] = df[col].fillna(sales_fallback)

    train_df["AvgUnitPrice"] = train_df["AvgUnitPrice"].fillna(price_fallback)
    for df in other_dfs:
        df["AvgUnitPrice"] = df["AvgUnitPrice"].fillna(price_fallback)

    # Calendar columns should already be complete; this is a defensive check.
    for col in ["DayOfWeek", "Month", "Year"]:
        train_df[col] = train_df[col].fillna(train_df[col].median())
        for df in other_dfs:
            df[col] = df[col].fillna(train_df[col].median())

    return train_df, other_dfs

inner_train_prepared, [validation_prepared] = prepare_feature_partitions(
    inner_train, [validation], feature_columns
)

X_inner_train = inner_train_prepared[feature_columns]
y_inner_train = inner_train_prepared[target_column]

X_validation = validation_prepared[feature_columns]
y_validation = validation_prepared[target_column]

print("NaN in X_inner_train:", X_inner_train.isna().sum().sum())
print("NaN in X_validation:", X_validation.isna().sum().sum())
print("NaN in y_inner_train:", y_inner_train.isna().sum())
print("NaN in y_validation:", y_validation.isna().sum())

assert X_inner_train.isna().sum().sum() == 0
assert X_validation.isna().sum().sum() == 0
assert y_inner_train.isna().sum() == 0
assert y_validation.isna().sum() == 0

## 8. Establish validation baselines

The baselines are calculated using the inner training partition and evaluated on the validation partition. This makes the model comparisons fair.

In [ ]:
# Level 0 — Global mean of training target
level0_value = y_inner_train.mean()
level0_validation_pred = np.full(len(y_validation), level0_value)

# Level 1 — Product-specific historical mean of the target
product_target_mean = (
    inner_train_prepared.groupby("StockCode")[target_column].mean()
)

level1_validation_pred = (
    validation_prepared["StockCode"]
    .map(product_target_mean)
    .fillna(level0_value)
    .to_numpy()
)

baseline_validation_results = pd.DataFrame({
    "Model": ["Level 0 - Global Mean", "Level 1 - Product Historical Mean"],
    "Validation_MAE": [
        mean_absolute_error(y_validation, level0_validation_pred),
        mean_absolute_error(y_validation, level1_validation_pred)
    ]
})

baseline_validation_results

## 9. Define experiment configurations

Each experiment changes a controlled set of Random Forest parameters. The first experiment reproduces the V3 configuration as the comparison reference.

**Runtime note:** Random Forest is CPU-based in this scikit-learn workflow. More trees and deeper trees can take considerably longer and use more memory.

In [ ]:
EXPERIMENTS = [
    {
        "Experiment": "RF-BASELINE",
        "n_estimators": 100,
        "max_depth": 15,
        "min_samples_leaf": 2
    },
    {
        "Experiment": "RF-200-TREES",
        "n_estimators": 200,
        "max_depth": 15,
        "min_samples_leaf": 2
    },
    {
        "Experiment": "RF-DEPTH-10",
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_leaf": 2
    },
    {
        "Experiment": "RF-DEPTH-20",
        "n_estimators": 100,
        "max_depth": 20,
        "min_samples_leaf": 2
    },
    {
        "Experiment": "RF-LEAF-5",
        "n_estimators": 100,
        "max_depth": 15,
        "min_samples_leaf": 5
    }
]

experiment_plan = pd.DataFrame(EXPERIMENTS)
experiment_plan

## 10. Run experiments on validation data only

The validation set is used to compare configurations. The final test set is not accessed in this section.

If runtime or memory is a problem, run only `RF-BASELINE` and `RF-DEPTH-10` first by temporarily reducing the `EXPERIMENTS` list.

In [ ]:
experiment_records = []
trained_validation_models = {}

for config in EXPERIMENTS:
    name = config["Experiment"]
    print(f"\nTraining {name} ...")

    model = RandomForestRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    )

    start_time = time.time()
    model.fit(X_inner_train, y_inner_train)
    fit_seconds = time.time() - start_time

    validation_pred = model.predict(X_validation)

    mae = mean_absolute_error(y_validation, validation_pred)
    rmse = np.sqrt(mean_squared_error(y_validation, validation_pred))
    r2 = r2_score(y_validation, validation_pred)

    experiment_records.append({
        "Experiment": name,
        "n_estimators": config["n_estimators"],
        "max_depth": config["max_depth"],
        "min_samples_leaf": config["min_samples_leaf"],
        "Validation_MAE": mae,
        "Validation_RMSE": rmse,
        "Validation_R2": r2,
        "Training_Time_Minutes": fit_seconds / 60
    })

    trained_validation_models[name] = model

    print(f"{name} | MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f} | time={fit_seconds/60:.2f} min")

experiment_results = (
    pd.DataFrame(experiment_records)
    .sort_values("Validation_MAE")
    .reset_index(drop=True)
)

experiment_results

## 11. Select the best configuration

Selection is based on **lowest validation MAE**, not final test MAE.

In [ ]:
best_experiment_row = experiment_results.iloc[0]
best_experiment_name = best_experiment_row["Experiment"]

best_config = next(
    config for config in EXPERIMENTS
    if config["Experiment"] == best_experiment_name
)

print("Best validation experiment:", best_experiment_name)
print("Best validation MAE:", round(best_experiment_row["Validation_MAE"], 4))
print("Selected parameters:")
print(best_config)

## 12. Refit selected model on all development data

After selecting parameters, we combine the inner training and validation periods into the full development dataset (ending September 12). The seven-day final purge remains untouched.

We then evaluate this refitted model once on the final test period beginning September 20.

In [ ]:
# Prepare full development and final test using development-only fallback statistics
development_prepared, [final_test_prepared] = prepare_feature_partitions(
    development, [final_test], feature_columns
)

X_development = development_prepared[feature_columns]
y_development = development_prepared[target_column]

X_final_test = final_test_prepared[feature_columns]
y_final_test = final_test_prepared[target_column]

print("Development rows:", len(development_prepared))
print("Final test rows:", len(final_test_prepared))
print("NaN X development:", X_development.isna().sum().sum())
print("NaN X final test:", X_final_test.isna().sum().sum())
print("NaN y development:", y_development.isna().sum())
print("NaN y final test:", y_final_test.isna().sum())

assert X_development.isna().sum().sum() == 0
assert X_final_test.isna().sum().sum() == 0
assert y_development.isna().sum() == 0
assert y_final_test.isna().sum() == 0

In [ ]:
best_model = RandomForestRegressor(
    n_estimators=best_config["n_estimators"],
    max_depth=best_config["max_depth"],
    min_samples_leaf=best_config["min_samples_leaf"],
    random_state=42,
    n_jobs=-1
)

print("Training selected model on all development data...")
final_fit_start = time.time()
best_model.fit(X_development, y_development)
final_fit_minutes = (time.time() - final_fit_start) / 60

print(f"Final model training completed in {final_fit_minutes:.2f} minutes.")

## 13. Final test evaluation

This is the final holdout evaluation. Do not change the model parameters based on these test results. If further tuning is required, create a new experiment plan and use a new untouched holdout period.

In [ ]:
final_test_prediction = best_model.predict(X_final_test)

final_rf_mae = mean_absolute_error(y_final_test, final_test_prediction)
final_rf_rmse = np.sqrt(mean_squared_error(y_final_test, final_test_prediction))
final_rf_r2 = r2_score(y_final_test, final_test_prediction)

# Recalculate final Level 0 and Level 1 on the same development/test split
final_level0_value = y_development.mean()
final_level0_prediction = np.full(len(y_final_test), final_level0_value)

final_product_mean = (
    development_prepared.groupby("StockCode")[target_column].mean()
)

final_level1_prediction = (
    final_test_prepared["StockCode"]
    .map(final_product_mean)
    .fillna(final_level0_value)
    .to_numpy()
)

final_level0_mae = mean_absolute_error(y_final_test, final_level0_prediction)
final_level1_mae = mean_absolute_error(y_final_test, final_level1_prediction)

final_comparison = pd.DataFrame({
    "Model": [
        "Level 0 - Global Mean",
        "Level 1 - Product Historical Mean",
        f"Optimized Random Forest ({best_experiment_name})"
    ],
    "Test_MAE": [
        final_level0_mae,
        final_level1_mae,
        final_rf_mae
    ],
    "Test_RMSE": [
        np.sqrt(mean_squared_error(y_final_test, final_level0_prediction)),
        np.sqrt(mean_squared_error(y_final_test, final_level1_prediction)),
        final_rf_rmse
    ],
    "Test_R2": [
        r2_score(y_final_test, final_level0_prediction),
        r2_score(y_final_test, final_level1_prediction),
        final_rf_r2
    ]
})

final_comparison

## 14. Calculate improvement over Level 1

A positive improvement means the selected Random Forest has lower MAE than Level 1.

In [ ]:
improvement_vs_level1 = (
    (final_level1_mae - final_rf_mae) / final_level1_mae
) * 100

print(f"Final Level 1 MAE: {final_level1_mae:.4f}")
print(f"Optimized RF MAE : {final_rf_mae:.4f}")
print(f"Improvement      : {improvement_vs_level1:.2f}%")

if final_rf_mae < final_level1_mae:
    print("RESULT: Optimized Random Forest beats Level 1 on the final test set.")
else:
    print("RESULT: Optimized Random Forest does not beat Level 1 on the final test set.")

## 15. Final test predictions and error analysis

In [ ]:
predictions = final_test_prepared[
    ["StockCode", "Date", target_column]
].copy()

predictions = predictions.rename(
    columns={target_column: "Actual_Future7DayDemand"}
)

predictions["Predicted_Future7DayDemand"] = final_test_prediction
predictions["Absolute_Error"] = (
    predictions["Actual_Future7DayDemand"] -
    predictions["Predicted_Future7DayDemand"]
).abs()

display(predictions.head(20))

print("\nAbsolute error summary:")
display(predictions["Absolute_Error"].describe())

print(
    "Zero-demand share in final test:",
    round((y_final_test == 0).mean() * 100, 2),
    "%"
)

## 16. Feature importance of selected model

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": best_model.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

display(feature_importance)

plt.figure(figsize=(9, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Optimized Random Forest — Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 17. Save experiment artifacts

Files are saved under the project's `results/` and `models/` folders.

In [ ]:
experiment_results.to_csv(
    RESULTS_DIR / "random_forest_experiment_results.csv",
    index=False
)

final_comparison.to_csv(
    RESULTS_DIR / "optimized_random_forest_final_comparison.csv",
    index=False
)

predictions.to_csv(
    RESULTS_DIR / "optimized_random_forest_test_predictions.csv",
    index=False
)

feature_importance.to_csv(
    RESULTS_DIR / "optimized_random_forest_feature_importance.csv",
    index=False
)

print("Experiment results and predictions saved.")

In [ ]:
model_artifact = {
    "model": best_model,
    "feature_columns": feature_columns,
    "target_column": target_column,
    "best_parameters": best_config,
    "validation_mae": float(best_experiment_row["Validation_MAE"]),
    "final_test_mae": float(final_rf_mae),
    "final_test_rmse": float(final_rf_rmse),
    "final_test_r2": float(final_rf_r2),
    "training_end": str(DEV_END.date()),
    "test_start": str(TEST_START.date()),
    "random_state": 42
}

MODEL_PATH = MODELS_DIR / "optimized_random_forest_demand_model.joblib"
joblib.dump(model_artifact, MODEL_PATH)

print("Selected model saved to:", MODEL_PATH)

## 18. Experiment conclusion

Complete this section after the notebook runs.

Record:
- Best validation experiment and parameters.
- Validation MAE.
- Final holdout test MAE, RMSE and R².
- Whether optimized Random Forest beats Level 1.
- Whether the final test MAE improves on the V3 reference MAE of 26.4215.

**Do not assume improvement before seeing the executed results.** If the final test result is worse, investigate rather than hiding or changing the test result.

### Step 4 completion criteria
- All experiments completed without errors.
- Hyperparameters selected using validation MAE only.
- Final test evaluated once after selection.
- Final metrics and predictions saved.
- Selected model artifact saved.
- Results documented for the project report.